# Domain 4: Prompt Engineering & Structured Output
## TS 4.3: Enforce Structured Output Using Tool Use and JSON Schemas
**Certification Focus**: CCAR-F Domain 4 (20% Exam Weight) | **Scenario Alignment**: Scenario 6 (Structured Data Extraction & Auditing)

---

### 🎯 Key Architectural Principles
1. **Tool Use for Structured Output**: Invoking a dummy/extraction tool (e.g. `extract_invoice`) with a strict `input_schema` is significantly more reliable than asking Claude to emit raw JSON in message text.
2. **Programmatic Validation**: Model-extracted payloads must be verified by deterministic Python code (cross-checking arithmetic, validating mandatory conditional fields).
3. **Validation Retry Loop**: When validation detects errors (e.g. arithmetic discrepancies), the specific errors are returned to Claude in a retry turn, enabling precise self-correction.

In [ ]:
# Step 1: Environment & Setup
import json
import sys
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from lib.mock_claude import MockClaudeClient, MockContentBlock, MockMessageResponse

print(" Environment initialized for TS 4.3 Structured Output via Tool Calling.")

### 📐 Step 2: Extraction Schema & Programmatic Validator
We define the extraction schema and the validation logic enforcing arithmetic consistency and category constraints.

In [ ]:
# Step 2: Schema & Validation Logic
EXTRACTION_TOOL = {
    "name": "extract_invoice",
    "description": "Extracts structured invoice fields and line items from document text.",
    "input_schema": {
        "type": "object",
        "properties": {
            "invoice_number": {"type": "string"},
            "vendor_name": {"type": "string"},
            "purchase_category": {"type": "string", "enum": ["hardware", "software_subscription", "other"]},
            "other_category_detail": {"type": ["string", "null"]},
            "line_items": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "description": {"type": "string"},
                        "quantity": {"type": "integer"},
                        "unit_price": {"type": "number"},
                        "total": {"type": "number"},
                    },
                    "required": ["description", "quantity", "unit_price", "total"],
                },
            },
            "stated_total": {"type": "number"},
        },
        "required": ["invoice_number", "vendor_name", "purchase_category", "line_items", "stated_total"],
    },
}

def validate_extraction(data: Dict[str, Any]) -> Tuple[bool, List[str]]:
    """Validates mathematical consistency and conditional field rules."""
    errors = []
    
    # 1. Arithmetic validation
    items = data.get("line_items", [])
    calc_sum = sum(item.get("total", 0.0) for item in items)
    stated = data.get("stated_total", 0.0)
    if abs(calc_sum - stated) > 0.01:
        errors.append(f"Arithmetic Mismatch: Sum of line items ({calc_sum:.2f}) does not match stated invoice total ({stated:.2f}).")

    # 2. Conditional category validation
    cat = data.get("purchase_category")
    if cat == "other" and not data.get("other_category_detail"):
        errors.append("Validation Error: 'other_category_detail' must be provided when purchase_category is 'other'.")

    return len(errors) == 0, errors

print(" Extraction schema and programmatic validator ready.")

### 🧪 Step 3: Interactive Simulation
We test:
1. Mismatched arithmetic -> Caught by validator.
2. Missing conditional category detail -> Caught by validator.
3. Correct payload -> Passes validation.

In [ ]:
# Step 3: Simulation
# 1. Invalid payload (math mismatch)
invalid_payload = {
    "invoice_number": "INV-001",
    "vendor_name": "Acme Corp",
    "purchase_category": "hardware",
    "line_items": [{"description": "Monitor", "quantity": 1, "unit_price": 200.0, "total": 200.0}],
    "stated_total": 350.0,  # Mismatch!
}
ok1, errors1 = validate_extraction(invalid_payload)
print("Math Mismatch Valid:", ok1)
print("Errors Detected:", errors1)

# 2. Valid payload
valid_payload = {
    "invoice_number": "INV-002",
    "vendor_name": "Cloud Services",
    "purchase_category": "software_subscription",
    "line_items": [{"description": "API Subscription", "quantity": 1, "unit_price": 100.0, "total": 100.0}],
    "stated_total": 100.0,
}
ok2, errors2 = validate_extraction(valid_payload)
print("\nValid Payload Result:", ok2)

### 🔬 Step 4: Verification Assertions
We assert that:
1. Validator catches arithmetic discrepancy.
2. Validator catches missing `other_category_detail`.
3. Valid invoice passes without errors.

In [ ]:
# Step 4: Verification Checks
assert ok1 is False
assert any("does not match stated invoice total" in e for e in errors1)

assert ok2 is True
assert len(errors2) == 0

# Check conditional other category
bad_cat = {"purchase_category": "other", "other_category_detail": None, "line_items": [], "stated_total": 0.0}
ok_cat, err_cat = validate_extraction(bad_cat)
assert ok_cat is False
assert any("other_category_detail" in e for e in err_cat)

print(" All verification assertions passed! Structured output validation verified.")

### 💡 CCAR-F Exam Golden Rules
- **Rule 1**: Use **tool calling schemas** for guaranteed JSON structure instead of parsing markdown JSON blocks.
- **Rule 2**: Programmatically validate extracted math (sum of line items vs stated total).
- **Rule 3**: Feed specific programmatic validation errors back into Claude for targeted self-correction.